In [ ]:
import json
import numpy as np
import torch
from pathlib import Path
from collections import Counter
from dataclasses import dataclass, field
from transformers import GPT2LMHeadModel
from IPython.display import Audio, display
import matplotlib.pyplot as plt

# Paths
MODEL_DIR  = Path('../models/distilgpt2_53tet/final')
VOCAB_DIR  = Path('../dataset/tokenized_hierarchical/full')
OUTPUT_DIR = Path('../output/generated_midi')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load vocabulary
vocab  = json.loads((VOCAB_DIR / 'vocab.json').read_text())
tok2id = vocab['tok2id']
id2tok = {int(k): v for k, v in vocab['id2tok'].items()}
VOCAB_SIZE = len(tok2id)

print(f'Device:     {DEVICE}')
print(f'Vocab size: {VOCAB_SIZE}')
print(f'Model dir:  {MODEL_DIR}')

## 1 — Load Trained Model

In [ ]:
model = GPT2LMHeadModel.from_pretrained(str(MODEL_DIR))
model.to(DEVICE)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded: {n_params:,} parameters')
print(f'Context:      {model.config.n_positions} tokens')
print(f'Layers:       {model.config.n_layer}')
print(f'Heads:        {model.config.n_head}')
print(f'Embed dim:    {model.config.n_embd}')

## 2 — `SongGenerator`: Autoregressive Token Sampler

Encapsulates all generation logic: unconditional, style-conditioned,
and prompt-seeded sampling with top-k / top-p / temperature control.

In [ ]:
@dataclass
class GenerationConfig:
    """Sampling hyperparameters for autoregressive generation."""
    max_tokens: int = 512
    temperature: float = 0.9
    top_k: int = 50
    top_p: float = 0.95
    repetition_penalty: float = 1.1


class SongGenerator:
    """Autoregressive token sampler for 53-TET chord progressions.

    Supports three generation modes:
        - Unconditional: <START> only
        - Style-conditioned: <START> STYLE_x
        - Prompt-seeded: arbitrary token prefix
    """

    def __init__(self, model, tok2id, id2tok, device='cpu'):
        self.model  = model
        self.tok2id = tok2id
        self.id2tok = id2tok
        self.device = device
        self.end_id = tok2id.get('<END>', 0)

    def _encode(self, tokens):
        ids = [self.tok2id[t] for t in tokens if t in self.tok2id]
        return torch.tensor([ids], dtype=torch.long, device=self.device)

    def generate(self, prompt_tokens, config=None):
        if config is None:
            config = GenerationConfig()
        input_ids = self._encode(prompt_tokens)
        with torch.no_grad():
            output = self.model.generate(
                input_ids,
                max_new_tokens=config.max_tokens,
                temperature=config.temperature,
                top_k=config.top_k,
                top_p=config.top_p,
                repetition_penalty=config.repetition_penalty,
                do_sample=True,
                pad_token_id=self.end_id,
            )
        tokens = [self.id2tok.get(int(i), '?') for i in output[0]]
        if '<END>' in tokens:
            tokens = tokens[:tokens.index('<END>') + 1]
        return tokens

    def unconditional(self, config=None):
        return self.generate(['<START>'], config)

    def style_conditioned(self, style, config=None):
        tok = f'STYLE_{style}'
        if tok not in self.tok2id:
            raise ValueError(f'{tok} not in vocabulary. '
                             f'Available: {self.available_styles()}')
        return self.generate(['<START>', tok], config)

    def prompt_seeded(self, seed_tokens, config=None):
        return self.generate(seed_tokens, config)

    def available_styles(self):
        return sorted(t.replace('STYLE_', '')
                      for t in self.tok2id if t.startswith('STYLE_'))


generator = SongGenerator(model, tok2id, id2tok, DEVICE)
print(f'Available styles: {generator.available_styles()}')

## 3 — Structural Validator

Verifies that generated sequences respect the token hierarchy
before attempting MIDI conversion.

In [ ]:
def validate_structure(tokens):
    """Check structural integrity of a generated token sequence."""
    checks = {
        'has_start':     tokens[0] == '<START>' if tokens else False,
        'has_end':       tokens[-1] == '<END>' if tokens else False,
        'has_style':     any(t.startswith('STYLE_') for t in tokens),
        'chord_balance': tokens.count('<CHORD_START>') == tokens.count('<CHORD_END>'),
        'midi_balance':  tokens.count('<MIDI_START>') == tokens.count('<MIDI_END>'),
        'n_chords':      tokens.count('<CHORD_START>'),
        'n_tokens':      len(tokens),
    }
    checks['valid'] = all(checks[k] for k in
                          ['has_start', 'has_end', 'has_style',
                           'chord_balance', 'midi_balance'])
    return checks


def pretty_print(tokens, max_chords=6):
    """Display a tokenized song with hierarchical indentation."""
    shown = 0
    for t in tokens:
        if t == '<START>':
            print(t)
        elif t.startswith('STYLE_'):
            print(f'  {t}')
        elif t == '<CHORD_START>':
            if shown >= max_chords:
                remaining = tokens.count('<CHORD_START>') - shown
                print(f'  ... ({remaining} more chords)')
                print('<END>')
                return
            print(f'  {t}')
        elif t == '<MIDI_START>':
            print(f'      {t}', end=' ')
        elif t == '<MIDI_END>':
            print(t)
        elif t == '<CHORD_END>':
            print(f'  {t}')
            shown += 1
        elif t == '<END>':
            print(t)
        elif t.startswith('MPE_'):
            print(t, end=' ')
        else:
            print(f'    {t}')

print('Validator loaded.')

## 4 — `TokenToMIDI`: 53-TET MPE Decoder

Parses the token stream into structured chord data and writes standard
MIDI files with **per-note pitch bend** for 53-TET microtonal accuracy.

**Pitch bend math:**
- 53-TET step = 1200 / 53 ≈ 22.64 cents
- MPE bend range = ±2 semitones = ±200 cents
- MIDI pitch wheel = [−8192, +8191] over ±2 semitones
- Per step: Δ_bend = steps × (22.64 / 200) × 8191

**MPE DAW compatibility (critical fix):**
- Channel 0 sends **MCM** (CC 127 = 15) to declare an MPE Lower Zone
- Channels 1–15 each carry one note + independent pitch bend
- RPN 0 on every member channel locks bend range to ±2 semitones
- RPN Null (101=127, 100=127) prevents accidental edits
- Pitch bend resets to center after every note-off

In [ ]:
from mido import MidiFile, MidiTrack, Message, MetaMessage


@dataclass
class ChordEvent:
    """Parsed chord from the token stream."""
    root: int = 0
    pattern: str = ''
    pl: float = 0.0
    dur_ticks: int = 480
    notes: list = field(default_factory=list)


class TokenToMIDI:
    """Decode 53-TET token sequences into MPE-MIDI files.

    Writes proper MPE with MCM on ch 0, per-channel pitch bend,
    RPN-locked bend range, and pitch bend reset after note-off.
    """

    CENTS_PER_53TET_STEP = 1200.0 / 53.0   # ~22.64 cents

    def __init__(self, ticks_per_beat=480, bpm=120, bend_range_semitones=2):
        self.ticks_per_beat = ticks_per_beat
        self.bpm = bpm
        self.bend_range_semitones = bend_range_semitones
        self.bend_range_cents = bend_range_semitones * 100.0
        self.velocity = 80

    def _bend_steps_to_wheel(self, steps):
        """53-TET bend steps --> MIDI pitch wheel [-8192, +8191]."""
        cents = steps * self.CENTS_PER_53TET_STEP
        ratio = cents / self.bend_range_cents
        wheel = int(round(ratio * 8191))
        return max(-8192, min(8191, wheel))

    def parse_tokens(self, tokens):
        """Parse flat token list --> list[ChordEvent]."""
        chords = []
        i = 0
        while i < len(tokens):
            if tokens[i] == '<CHORD_START>':
                chord = ChordEvent()
                i += 1
                while i < len(tokens) and tokens[i] != '<MIDI_START>':
                    t = tokens[i]
                    if t.startswith('ROOT_'):
                        chord.root = int(t.split('_', 1)[1])
                    elif t.startswith('PATTERN_'):
                        chord.pattern = t.split('_', 1)[1]
                    elif t.startswith('PL_'):
                        parts = t.split('_')
                        try:
                            chord.pl = float(f'{parts[1]}.{parts[2]}')
                        except (IndexError, ValueError):
                            pass
                    elif t.startswith('DUR_'):
                        chord.dur_ticks = int(t.split('_', 1)[1])
                    i += 1
                if i < len(tokens) and tokens[i] == '<MIDI_START>':
                    i += 1
                    while i < len(tokens) and tokens[i] != '<MIDI_END>':
                        if tokens[i].startswith('MPE_'):
                            parts = tokens[i].split('_')
                            midi_note = int(parts[1])
                            b = parts[2]
                            if b == '0':
                                bend = 0
                            elif b.startswith('p'):
                                bend = int(b[1:])
                            elif b.startswith('m'):
                                bend = -int(b[1:])
                            else:
                                bend = 0
                            chord.notes.append((midi_note, bend))
                        i += 1
                while i < len(tokens) and tokens[i] in ('<MIDI_END>', '<CHORD_END>'):
                    i += 1
                if chord.notes:
                    chords.append(chord)
                continue
            i += 1
        return chords

    def _write_mpe_header(self, track):
        """MCM on ch 0 + RPN pitch-bend-range on ch 1-15 + RPN Null."""
        # MCM: declare MPE Lower Zone with 15 member channels
        track.append(Message('control_change', channel=0,
                             control=127, value=15, time=0))
        for ch in range(1, 16):
            # RPN 0: pitch bend sensitivity
            track.append(Message('control_change', channel=ch,
                                 control=101, value=0, time=0))
            track.append(Message('control_change', channel=ch,
                                 control=100, value=0, time=0))
            track.append(Message('control_change', channel=ch,
                                 control=6, value=self.bend_range_semitones, time=0))
            track.append(Message('control_change', channel=ch,
                                 control=38, value=0, time=0))
            # RPN Null -- prevent accidental edits
            track.append(Message('control_change', channel=ch,
                                 control=101, value=127, time=0))
            track.append(Message('control_change', channel=ch,
                                 control=100, value=127, time=0))

    def chords_to_midi(self, chords, output_path):
        """Write ChordEvent list --> MPE-MIDI file."""
        mid   = MidiFile(ticks_per_beat=self.ticks_per_beat)
        track = MidiTrack()
        mid.tracks.append(track)

        track.append(MetaMessage('set_tempo',
                                 tempo=int(60_000_000 / self.bpm), time=0))
        self._write_mpe_header(track)

        for chord in chords:
            if not chord.notes:
                continue
            # Note-on: pitch bend then note_on, each on its own channel
            for idx, (note, bend_steps) in enumerate(chord.notes):
                ch = (idx % 15) + 1
                wheel = self._bend_steps_to_wheel(bend_steps)
                track.append(Message('pitchwheel', channel=ch,
                                     pitch=wheel, time=0))
                track.append(Message('note_on', channel=ch,
                                     note=note, velocity=self.velocity, time=0))
            # Note-off: first carries the duration delta
            first = True
            for idx, (note, _) in enumerate(chord.notes):
                ch = (idx % 15) + 1
                dt = chord.dur_ticks if first else 0
                track.append(Message('note_off', channel=ch,
                                     note=note, velocity=0, time=dt))
                # Reset pitch bend to center after release
                track.append(Message('pitchwheel', channel=ch,
                                     pitch=0, time=0))
                first = False

        track.append(MetaMessage('end_of_track', time=0))
        out = Path(output_path)
        out.parent.mkdir(parents=True, exist_ok=True)
        mid.save(str(out))
        return out

    def tokens_to_midi(self, tokens, output_path):
        """Full pipeline: tokens --> parse --> MIDI."""
        chords = self.parse_tokens(tokens)
        return self.chords_to_midi(chords, output_path)


decoder = TokenToMIDI(ticks_per_beat=480, bpm=120, bend_range_semitones=2)
print(f'TokenToMIDI ready -- {decoder.CENTS_PER_53TET_STEP:.2f} cents/step, '
      f'+/-{decoder.bend_range_cents:.0f} cents bend range')
print('MPE: MCM on ch 0, member channels 1-15, RPN-null locked')

## 5 — Microtonal Audio Engine

Direct additive synthesis at exact 53-TET frequencies, bypassing any
DAW or soundfont limitations.  The inline ▶ player lets you hear
microtonal content without leaving the notebook.

In [ ]:
class MicrotonalSynth:
    """Synthesize 53-TET chord events with additive sine waves."""

    def __init__(self, sample_rate=44100):
        self.sr = sample_rate

    def _freq(self, midi_note, bend_steps):
        """MIDI note + 53-TET bend --> frequency (Hz)."""
        base  = 440.0 * 2 ** ((midi_note - 69) / 12.0)
        cents = bend_steps * (1200.0 / 53.0)
        return base * 2 ** (cents / 1200.0)

    def render(self, chords, bpm=120):
        """Render ChordEvent list --> float32 audio array."""
        segments = []
        for chord in chords:
            dur_sec  = (chord.dur_ticks / 480.0) * (60.0 / bpm)
            n_samps  = max(1, int(dur_sec * self.sr))
            t = np.linspace(0, dur_sec, n_samps, endpoint=False)

            wave = np.zeros(n_samps, dtype=np.float64)
            for note, bend in chord.notes:
                f = self._freq(note, bend)
                wave += np.sin(2 * np.pi * f * t)
                wave += 0.25 * np.sin(2 * np.pi * 2 * f * t)
                wave += 0.08 * np.sin(2 * np.pi * 3 * f * t)

            if chord.notes:
                wave /= len(chord.notes)

            # Smooth envelope (attack / release)
            att = min(int(0.012 * self.sr), n_samps // 4)
            rel = min(int(0.060 * self.sr), n_samps // 3)
            env = np.ones(n_samps)
            if att > 0:
                env[:att] = np.linspace(0, 1, att)
            if rel > 0:
                env[-rel:] = np.linspace(1, 0, rel)
            wave *= env
            segments.append(wave)

        if not segments:
            return np.zeros(self.sr, dtype=np.float32)

        audio = np.concatenate(segments)
        peak  = np.max(np.abs(audio))
        if peak > 0:
            audio *= 0.8 / peak
        return audio.astype(np.float32)

    def play(self, chords, bpm=120):
        """Render and return an inline Audio widget."""
        audio = self.render(chords, bpm)
        return Audio(audio, rate=self.sr)


synth = MicrotonalSynth()
print(f'MicrotonalSynth ready -- {synth.sr} Hz sample rate')


def generate_and_play(gen_func, label, filename, cfg=None, bpm=120):
    """Run a generation function, save MIDI, and play inline audio.

    Returns (tokens, chords) tuple.
    """
    tokens = gen_func(cfg) if cfg else gen_func()
    v = validate_structure(tokens)
    ok = '✓' if v['valid'] else '✗'
    print(f"{ok} {label}: {v['n_chords']} chords, {v['n_tokens']} tok "
          f"| balanced={v['chord_balance'] and v['midi_balance']}")

    chords = decoder.parse_tokens(tokens)
    path = decoder.chords_to_midi(chords, OUTPUT_DIR / filename)
    print(f'  -> MIDI: {path}')

    display(synth.play(chords, bpm))
    return tokens, chords

## 6 — Model Diagnostics

**Is the model actually generating microtonal content?**
This cell generates samples and reports:
- Token category distribution
- Pitch bend histogram (non-zero bends = microtonality)
- Root note spread and dissonance trajectory

In [ ]:
def analyze_generation(tokens, label=''):
    """Comprehensive analysis of a generated token sequence."""
    cats = Counter()
    bends, roots, patterns, pls = [], [], [], []

    for t in tokens:
        if t in ('<START>', '<END>', '<CHORD_START>', '<CHORD_END>',
                 '<MIDI_START>', '<MIDI_END>'):
            cats['structural'] += 1
        elif t.startswith('STYLE_'):   cats['style']   += 1
        elif t.startswith('ROOT_'):
            cats['root'] += 1
            roots.append(int(t.split('_')[1]))
        elif t.startswith('PATTERN_'):
            cats['pattern'] += 1
            patterns.append(t.split('_', 1)[1])
        elif t.startswith('PL_'):
            cats['pl'] += 1
            parts = t.split('_')
            try: pls.append(float(f'{parts[1]}.{parts[2]}'))
            except: pass
        elif t.startswith('DUR_'):     cats['dur']     += 1
        elif t.startswith('MPE_'):
            cats['mpe'] += 1
            b = t.split('_')[2]
            if   b == '0':            bends.append(0)
            elif b.startswith('p'):   bends.append(int(b[1:]))
            elif b.startswith('m'):   bends.append(-int(b[1:]))
        else:
            cats['other'] += 1

    sep = '═' * 50
    title = f'  Analysis: {label}' if label else '  Generation Analysis'
    print(f'{sep}')
    print(title)
    print(f'{sep}')
    print(f'  Total tokens: {len(tokens)} | Chords: {tokens.count("<CHORD_START>")}')
    print()

    print('  Token categories:')
    for cat, cnt in cats.most_common():
        pct = cnt / max(len(tokens), 1) * 100
        print(f'    {cat:>12s}: {cnt:>5d}  ({pct:5.1f}%)')
    print()

    # KEY DIAGNOSTIC: Is the output microtonal?
    if bends:
        nz = [b for b in bends if b != 0]
        nz_pct = len(nz) / len(bends) * 100
        print(f'  Microtonal analysis:')
        print(f'    MPE notes total : {len(bends)}')
        print(f'    Non-zero bends  : {len(nz)} ({nz_pct:.1f}%)')
        if nz:
            cents = [b * 22.64 for b in nz]
            print(f'    Step range      : [{min(nz):+d}, {max(nz):+d}]')
            print(f'    Cents range     : [{min(cents):+.1f}, {max(cents):+.1f}]')
            print(f'    Mean |bend|     : {np.mean(np.abs(nz)):.2f} steps '
                  f'({np.mean(np.abs(cents)):.1f} cents)')
        else:
            print(f'    WARNING: Zero microtonal bends -- output is effectively 12-TET.')
            print(f'    The model may not have learned microtonal patterns.')
    print()

    # Plots
    n_plots = sum([bool(bends), bool(roots), bool(pls)])
    if n_plots == 0:
        return {'bends': bends, 'roots': roots, 'patterns': patterns, 'pl': pls}

    fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 4))
    if n_plots == 1:
        axes = [axes]
    idx = 0

    if bends:
        lo, hi = min(bends) - 1, max(bends) + 2
        axes[idx].hist(bends, bins=range(lo, hi), color='teal',
                       alpha=0.7, edgecolor='black', linewidth=0.5)
        axes[idx].axvline(0, color='red', ls='--', lw=1, label='12-TET center')
        axes[idx].set_xlabel('Bend (53-TET steps)')
        axes[idx].set_ylabel('Count')
        axes[idx].set_title('Pitch Bend Distribution')
        axes[idx].legend(fontsize=8)
        idx += 1

    if roots:
        axes[idx].hist(roots, bins=range(54), color='coral', alpha=0.7)
        axes[idx].set_xlabel('Root (0-52)')
        axes[idx].set_ylabel('Count')
        axes[idx].set_title('Root Note Distribution')
        idx += 1

    if pls:
        axes[idx].fill_between(range(len(pls)), pls, alpha=0.25, color='teal')
        axes[idx].plot(pls, color='teal', lw=1.2)
        axes[idx].axhline(np.mean(pls), color='red', ls='--', lw=1,
                          label=f'μ={np.mean(pls):.1f}')
        axes[idx].set_xlabel('Chord Index')
        axes[idx].set_ylabel('PL Dissonance')
        axes[idx].set_title('Harmonic Tension')
        axes[idx].legend(fontsize=8)

    plt.tight_layout()
    plt.show()
    return {'bends': bends, 'roots': roots, 'patterns': patterns, 'pl': pls}


# Run diagnostics on 3 quick samples
print('Generating 3 diagnostic samples...\n')
cfg_diag = GenerationConfig(max_tokens=400, temperature=0.9, top_k=50)
for i in range(3):
    diag_tokens = generator.unconditional(cfg_diag)
    analyze_generation(diag_tokens, label=f'Sample {i+1}')
    print()

## 7 — Unconditional Generation

Generate a complete song from scratch — the model freely selects style
and content.  Press **▶** on the audio widget to listen.

In [ ]:
cfg_free = GenerationConfig(max_tokens=400, temperature=0.9, top_k=50)
tokens_free, chords_free = generate_and_play(
    generator.unconditional, 'Unconditional', 'unconditional.mid', cfg_free
)
print()
pretty_print(tokens_free, max_chords=4)

## 8 — Style-Conditioned Generation

Force the model to write in a specific genre.  Each style gets its own
MIDI file and inline audio player.

In [ ]:
styles_to_test = ['Jazz', 'Bossa', 'Blues']
cfg_style = GenerationConfig(max_tokens=400, temperature=0.85, top_k=40)

for style in styles_to_test:
    if f'STYLE_{style}' not in tok2id:
        print(f'STYLE_{style} not in vocabulary -- skipping.\n')
        continue
    tokens_s, _ = generate_and_play(
        lambda c=cfg_style, s=style: generator.style_conditioned(s, c),
        f'STYLE_{style}', f'style_{style.lower()}.mid', bpm=120
    )
    pretty_print(tokens_s, max_chords=3)
    print()

## 9 — Prompt-Seeded (Dissonant) Generation

Seed the model with a dissonant chord cluster and let it resolve
the harmonic progression autonomously.

In [ ]:
seed = [
    '<START>', 'STYLE_Jazz',
    '<CHORD_START>', 'ROOT_0', 'PATTERN_pythagorean', 'PL_35_2', 'DUR_960',
    '<MIDI_START>',
    'MPE_48_0', 'MPE_54_p3', 'MPE_60_m2', 'MPE_66_p1', 'MPE_71_m3',
    '<MIDI_END>', '<CHORD_END>',
]
seed_valid = [t for t in seed if t in tok2id]
print(f'Seed ({len(seed_valid)} tokens): {" ".join(seed_valid[:15])}...')

tokens_seed, chords_seed = generate_and_play(
    lambda c=GenerationConfig(max_tokens=400, temperature=0.95, top_k=50):
        generator.prompt_seeded(seed_valid, c),
    'Seeded (dissonant)', 'seeded_dissonant.mid', bpm=120
)
print()
pretty_print(tokens_seed, max_chords=5)

## 10 — Batch Generation & Statistics

Generate multiple samples per style, aggregate structural validity,
and plot chord-count distributions.

In [ ]:
n_samples = 5
cfg_batch = GenerationConfig(max_tokens=400, temperature=0.9, top_k=50)
results = {}

for style in generator.available_styles()[:6]:
    samples = []
    for i in range(n_samples):
        gen = generator.style_conditioned(style, cfg_batch)
        v = validate_structure(gen)
        samples.append(v)
        decoder.tokens_to_midi(gen, OUTPUT_DIR / f'batch_{style.lower()}_{i}.mid')
    results[style] = samples

print(f"{'Style':<12s} {'Valid':>5s} {'Avg Chords':>10s} {'Avg Tokens':>10s}")
print('─' * 42)
for style, samps in results.items():
    n_valid = sum(1 for s in samps if s['valid'])
    avg_ch  = np.mean([s['n_chords'] for s in samps])
    avg_tk  = np.mean([s['n_tokens'] for s in samps])
    print(f'{style:<12s} {n_valid}/{n_samples:>3d}  {avg_ch:>10.1f} {avg_tk:>10.1f}')

fig, ax = plt.subplots(figsize=(10, 4))
for style, samps in results.items():
    chords = [s['n_chords'] for s in samps]
    ax.bar(style, np.mean(chords), yerr=np.std(chords), capsize=4, alpha=0.75)
ax.set_ylabel('Chords per Song')
ax.set_title(f'Generated Song Length by Style (n={n_samples} per style)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 11 — Dissonance Profile

Extract PL dissonance values from a longer generation and visualize
the harmonic tension trajectory over chord time.

In [ ]:
gen_long = generator.style_conditioned(
    'Jazz', GenerationConfig(max_tokens=800, temperature=0.9, top_k=50)
)
analysis = analyze_generation(gen_long, label='Long Jazz sample')

## 12 — Pitch Bend Verification

Round-trip sanity check: known intervals → MIDI → read back → verify.
Audible A/B comparison between 12-TET and 53-TET voicings.

In [ ]:
# Verify pitch bend mapping
print(f"{'Steps':>6s} {'Cents':>8s} {'Wheel':>7s} {'Ratio':>8s}")
print('─' * 34)
for s in [-4, -3, -2, -1, 0, 1, 2, 3, 4]:
    cents = s * decoder.CENTS_PER_53TET_STEP
    wheel = decoder._bend_steps_to_wheel(s)
    ratio = wheel / 8191 if wheel != 0 else 0.0
    print(f'{s:>6d} {cents:>8.2f} {wheel:>7d} {ratio:>8.4f}')

# Round-trip: known chord -> MIDI -> verify
test_tokens = [
    '<START>', 'STYLE_Jazz',
    '<CHORD_START>', 'ROOT_0', 'PATTERN_pythagorean', 'PL_18_4', 'DUR_960',
    '<MIDI_START>', 'MPE_48_0', 'MPE_55_p1', 'MPE_60_0', 'MPE_64_m1', 'MPE_67_p2',
    '<MIDI_END>', '<CHORD_END>', '<END>',
]
chords_test = decoder.parse_tokens(test_tokens)
print(f'\nParsed {len(chords_test)} chord(s):')
for c in chords_test:
    notes_str = ', '.join(
        f'MIDI {n} bend={b} (wheel={decoder._bend_steps_to_wheel(b)})'
        for n, b in c.notes)
    print(f'  ROOT={c.root} DUR={c.dur_ticks} PL={c.pl} [{notes_str}]')

test_path = decoder.chords_to_midi(chords_test, OUTPUT_DIR / 'test_roundtrip.mid')
print(f'\n-> Test MIDI: {test_path}')

# Audible A/B comparison
print('\n12-TET reference (all bends = 0):')
chords_12 = [ChordEvent(notes=[(48, 0), (55, 0), (60, 0), (64, 0), (67, 0)],
                         dur_ticks=960)]
display(synth.play(chords_12))

print('53-TET version (with microtonal bends):')
display(synth.play(chords_test))

---

## Notes

- Adjust `temperature` (0.7–1.0) to control creativity vs. coherence.
- Lower `top_k` (20–30) for more conservative harmonic choices.
- Increase `max_tokens` for longer progressions (up to model context window).
- The **MPE-MIDI** files now include MCM (Channel Mode Message) on channel 0, making
  them compatible with MPE-aware DAWs (Ableton Live 11+, Logic Pro, REAPER, Bitwig).
- In your DAW, set the synth’s **pitch bend range to ±2 semitones** for correct tuning.
- The inline **▶ audio player** uses additive synthesis at exact 53-TET frequencies —
  it always plays microtonal content correctly, independent of DAW or soundfont.